# 🧠 Letter Recognition (EMNIST) for 3D Neural Flow — augmented training

Trains the **784 → 64 → 32 → 26** network on **EMNIST-Letters** (124,800 train / 20,800 test handwritten letters) with heavy **GPU data augmentation**, so it copes with letters drawn by finger or mouse in the web app: different sizes, positions, slants, stroke thickness, and hard (binary) strokes.

### Instructions
1. Click **Runtime** > **Change runtime type** > select **T4 GPU**.
2. Click **Runtime** > **Run all**. Training takes a few minutes (set `EPOCHS` in step 5 to train longer).
3. At the end `mlp_weights.json` downloads automatically.
4. Replace `data/model_weights.json` in your `neural-flows` folder with it.

> The web app scales each drawn letter so its long side is `TARGET_SIZE` px (see `components/NeuralVisualizerApp.tsx`). Step 3b prints the real glyph size in EMNIST so you can check the two match.

In [ ]:
# Step 1: Check GPU + seeds
import torch
torch.manual_seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Training device: {device}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

In [ ]:
# Step 2: Download EMNIST-Letters (real handwritten letters)
import torchvision

print('Downloading EMNIST Letters dataset...')
train_dataset = torchvision.datasets.EMNIST(root='./data', split='letters', train=True, download=True)
test_dataset = torchvision.datasets.EMNIST(root='./data', split='letters', train=False, download=True)

print('Train samples:', len(train_dataset))
print('Test samples:', len(test_dataset))

In [ ]:
# Step 3: Load everything onto the GPU as tensors (fast augmentation, no DataLoader needed)
import numpy as np

def to_tensor(ds):
    # Raw EMNIST images are stored transposed -> swap the two image axes to make them upright
    x = ds.data.transpose(1, 2).float().div(255.0)
    y = ds.targets.long() - 1          # labels 1-26 -> 0-25 (A=0 ... Z=25)
    return x.to(device), y.to(device)

X_all, y_all = to_tensor(train_dataset)
X_test, y_test = to_tensor(test_dataset)

# Hold out 5% of the training set as a validation set (used to pick the best epoch)
g = torch.Generator().manual_seed(0)
perm = torch.randperm(len(X_all), generator=g).to(device)
n_val = len(X_all) // 20
val_idx, tr_idx = perm[:n_val], perm[n_val:]
X_val, y_val = X_all[val_idx], y_all[val_idx]
X_train, y_train = X_all[tr_idx], y_all[tr_idx]

# Normalisation constants (exported to the web app, which applies the same (x - mean) / std)
mean = X_train.mean().item()
std = X_train.std().item()
print(f'train={len(X_train)}  val={len(X_val)}  test={len(X_test)}')
print(f'mean={mean:.4f}  std={std:.4f}')

In [ ]:
# Step 3b: How big are EMNIST glyphs? The web app scales drawings to TARGET_SIZE to match this.
def bbox_long_side(x, thr=0.15):
    mask = x > thr
    rows, cols = mask.any(dim=2), mask.any(dim=1)
    idx = torch.arange(28, device=x.device)
    r_min = torch.where(rows, idx, torch.full_like(idx, 28)).min(1).values
    r_max = torch.where(rows, idx, torch.full_like(idx, -1)).max(1).values
    c_min = torch.where(cols, idx, torch.full_like(idx, 28)).min(1).values
    c_max = torch.where(cols, idx, torch.full_like(idx, -1)).max(1).values
    long_side = torch.maximum(r_max - r_min + 1, c_max - c_min + 1).float()
    return long_side[rows.any(1)]

sizes = bbox_long_side(X_train[:20000])
print(f'EMNIST glyph long side (px): median={sizes.median().item():.0f}  '
      f'p10={sizes.quantile(0.10).item():.0f}  p90={sizes.quantile(0.90).item():.0f}')
print('-> set TARGET_SIZE in components/NeuralVisualizerApp.tsx to about the median (currently 24).')

In [ ]:
# Step 4: GPU data augmentation that imitates hand-drawn input from the web app
import math
import torch.nn.functional as F

def augment(x):
    """x: (B, 28, 28) in [0, 1] on the GPU -> augmented (B, 28, 28)."""
    B = x.size(0)
    dev = x.device
    x = x.unsqueeze(1)
    rnd = lambda: torch.rand(B, device=dev) * 2 - 1      # uniform in [-1, 1]

    # Random affine: rotation (+-12 deg), anisotropic scale (+-18%), shear, translation (+-2.5 px)
    ang = rnd() * math.radians(12)
    sx = 1 + rnd() * 0.18
    sy = 1 + rnd() * 0.18
    shear = rnd() * 0.20
    cos, sin = torch.cos(ang), torch.sin(ang)
    theta = torch.zeros(B, 2, 3, device=dev)
    theta[:, 0, 0] = cos * sx
    theta[:, 0, 1] = -sin * sx + shear
    theta[:, 1, 0] = sin * sy
    theta[:, 1, 1] = cos * sy
    theta[:, 0, 2] = rnd() * (2.5 / 14)
    theta[:, 1, 2] = rnd() * (2.5 / 14)
    grid = F.affine_grid(theta, list(x.shape), align_corners=False)
    x = F.grid_sample(x, grid, mode='bilinear', padding_mode='zeros', align_corners=False)

    # Stroke thickness: dilate (thicker) or erode (thinner) a random subset
    r = torch.rand(B, 1, 1, 1, device=dev)
    dilated = F.max_pool2d(x, 3, 1, 1)
    eroded = -F.max_pool2d(-x, 3, 1, 1)
    x = torch.where(r < 0.30, dilated, torch.where(r < 0.45, eroded, x))

    # Blur some samples, binarise others (the app draws near-binary strokes)
    k = torch.tensor([[1., 2., 1.], [2., 4., 2.], [1., 2., 1.]], device=dev).div(16).view(1, 1, 3, 3)
    blurred = F.conv2d(x, k, padding=1)
    hard = torch.clamp((x - 0.2) * 5, 0, 1)
    r = torch.rand(B, 1, 1, 1, device=dev)
    x = torch.where(r < 0.25, blurred, torch.where(r < 0.55, hard, x))

    # Brightness jitter
    x = x * (0.85 + 0.15 * torch.rand(B, 1, 1, 1, device=dev))
    return x.clamp(0, 1).squeeze(1)

# Quick sanity check of the augmentations (original on top, augmented below)
try:
    import matplotlib.pyplot as plt
    sample = X_train[:12]
    aug = augment(sample)
    fig, axes = plt.subplots(2, 12, figsize=(14, 2.6))
    for i in range(12):
        axes[0, i].imshow(sample[i].cpu(), cmap='gray'); axes[0, i].axis('off')
        axes[1, i].imshow(aug[i].cpu(), cmap='gray'); axes[1, i].axis('off')
    plt.show()
except Exception as e:
    print('Preview skipped:', e)

In [ ]:
# Step 5: Train 784 -> 64 -> 32 -> 26 with augmentation (best epoch picked on the validation set)
import time
import torch.nn as nn

EPOCHS = 80        # raise this (e.g. 150) to train even longer
BATCH = 256
KEEP_CLEAN = 0.15  # fraction of each batch left un-augmented

model = nn.Sequential(
    nn.Linear(784, 64),
    nn.ReLU(),
    nn.Linear(64, 32),
    nn.ReLU(),
    nn.Linear(32, 26)
).to(device)

steps_per_epoch = math.ceil(len(X_train) / BATCH)
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-3, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.OneCycleLR(
    optimizer, max_lr=4e-3, epochs=EPOCHS, steps_per_epoch=steps_per_epoch, pct_start=0.1
)
criterion = nn.CrossEntropyLoss()

def evaluate(X, y, aug=False, seed=0):
    model.eval()
    if aug:
        cpu_state = torch.get_rng_state()
        cuda_state = torch.cuda.get_rng_state_all() if torch.cuda.is_available() else None
        torch.manual_seed(seed)   # same augmentations every time -> comparable numbers
    correct = 0
    with torch.no_grad():
        for i in range(0, len(X), 2048):
            xb = X[i:i + 2048]
            if aug:
                xb = augment(xb)
            logits = model((xb.reshape(len(xb), -1) - mean) / std)
            correct += (logits.argmax(1) == y[i:i + 2048]).sum().item()
    if aug:   # restore the training RNG so every epoch keeps getting fresh shuffles + augmentations
        torch.set_rng_state(cpu_state)
        if cuda_state is not None:
            torch.cuda.set_rng_state_all(cuda_state)
    return correct / len(X)

best_score, best_state, best_epoch = -1.0, None, 0
start_time = time.time()

for epoch in range(EPOCHS):
    model.train()
    order = torch.randperm(len(X_train), device=device)
    total_loss, correct = 0.0, 0
    for i in range(0, len(order), BATCH):
        idx = order[i:i + BATCH]
        xb, yb = X_train[idx], y_train[idx]
        xa = augment(xb)
        keep = torch.rand(len(xb), 1, 1, device=device) < KEEP_CLEAN
        xb = torch.where(keep, xb, xa)

        optimizer.zero_grad()
        logits = model((xb.reshape(len(xb), -1) - mean) / std)
        loss = criterion(logits, yb)
        loss.backward()
        optimizer.step()
        scheduler.step()

        total_loss += loss.item() * len(xb)
        correct += (logits.argmax(1) == yb).sum().item()

    val_clean = evaluate(X_val, y_val)
    val_aug = evaluate(X_val, y_val, aug=True)
    score = val_clean + val_aug
    if score > best_score:
        best_score, best_epoch = score, epoch + 1
        best_state = {k: v.detach().clone() for k, v in model.state_dict().items()}

    print(f'Epoch {epoch + 1:3d}/{EPOCHS} | loss {total_loss / len(X_train):.4f} | '
          f'train(aug) {correct / len(X_train) * 100:.1f}% | '
          f'val {val_clean * 100:.1f}% | val(aug) {val_aug * 100:.1f}%')

print(f'\nTraining complete in {time.time() - start_time:.0f}s! Best epoch: {best_epoch}')

In [ ]:
# Step 6: Final evaluation on the untouched test set (best epoch)
model.load_state_dict(best_state)
test_clean = evaluate(X_test, y_test)
test_aug = evaluate(X_test, y_test, aug=True)
print(f'Test accuracy (clean EMNIST):      {test_clean * 100:.2f}%')
print(f'Test accuracy (augmented, harder): {test_aug * 100:.2f}%')

# Which letters get confused?
model.eval()
with torch.no_grad():
    preds = model((X_test.reshape(len(X_test), -1) - mean) / std).argmax(1)
conf = torch.bincount(y_test * 26 + preds, minlength=26 * 26).reshape(26, 26).cpu()
letters = 'ABCDEFGHIJKLMNOPQRSTUVWXYZ'
per_letter = conf.diag().float() / conf.sum(1).float()
worst = per_letter.argsort()[:8]
print('\nWorst letters:   ' + '  '.join(f'{letters[i]} {per_letter[i] * 100:.0f}%' for i in worst))
off = conf.clone()
off.fill_diagonal_(0)
top = off.flatten().topk(10)
print('Top confusions:  ' + '  '.join(
    f'{letters[i // 26]}->{letters[i % 26]} ({int(n)})' for n, i in zip(top.values, top.indices)))

In [ ]:
# Step 7: Generate Clean Preset Letter Templates (A-Z)
from PIL import Image, ImageDraw, ImageFont

CLASSES = list('ABCDEFGHIJKLMNOPQRSTUVWXYZ')
templates = []

# Fallback font rendering
try:
    fnt = ImageFont.truetype("/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf", 20)
except:
    fnt = ImageFont.load_default()

for ci, ch in enumerate(CLASSES):
    im = Image.new('L', (28, 28), 0)
    d = ImageDraw.Draw(im)
    bb = d.textbbox((0, 0), ch, font=fnt)
    tw, th = bb[2] - bb[0], bb[3] - bb[1]
    cx, cy = (28 - tw) // 2 - bb[0], (28 - th) // 2 - bb[1]
    d.text((cx, cy), ch, fill=255, font=fnt)
    arr = np.array(im, dtype=np.float32) / 255.0
    templates.append(arr.flatten().tolist())

In [ ]:
# Step 8: Export model to JSON and auto-download
import json

def layer_dict(linear, activation):
    return {
        "activation": activation,
        "weights": np.round(linear.weight.detach().cpu().numpy(), 5).tolist(),
        "biases": np.round(linear.bias.detach().cpu().numpy(), 5).tolist(),
    }

export_dict = {
    "mode": "inference",
    "classes": CLASSES,
    "architecture": [784, 64, 32, 26],
    "normalization": {"mean": float(mean), "std": float(std)},
    "layers": [
        layer_dict(model[0], "relu"),
        layer_dict(model[2], "relu"),
        layer_dict(model[4], "linear"),
    ],
    "templates": templates
}

output_file = "mlp_weights.json"
with open(output_file, "w") as f:
    json.dump(export_dict, f)

print(f"Successfully saved {output_file}!")

# Download file to your browser
try:
    from google.colab import files
    files.download(output_file)
    print("Triggered automatic download of mlp_weights.json!")
except ImportError:
    print("Not in Colab - file saved locally.")